# Multi-Scale Lagged Recurrent Neural Network (MS-LRNN)

**A research-style, fully runnable PyTorch notebook for benchmarking a lagged/stacked recurrent architecture on language modeling.**

This notebook implements the architecture discussed in the prompt:

- each recurrent layer is replaced by multiple recurrent sublayers/branches;
- one branch is a normal RNN branch using lag `r = 1`;
- the other branches are lagged/dilated branches using hidden states from `r > 1` timesteps ago;
- all branch outputs are fused and passed upward to the next recurrent layer;
- stacked layers repeat the same design;
- ablations compare vanilla RNN, two-branch lagged RNN, multi-lag variants, gated fusion, tied branch weights, and GRU-based lagged variants.

The primary benchmark task is **token-level causal language modeling** on **WikiText-2**.

The primary metric is **validation/test perplexity**:

$\left[\text{PPL} = \exp(\text{cross-entropy})\right]$

For language modeling, perplexity is more appropriate than BLEU/ROUGE because we are evaluating next-token probability distributions, not comparing one generated sequence against one or more reference summaries/translations.

---

## What this notebook gives you

1. A precise PyTorch implementation of a stacked multi-scale lagged RNN.
2. WikiText-2 dataset loading and word-level tokenization.
3. Reproducible train/validation/test splits.
4. Evaluation metrics: cross-entropy, perplexity, token accuracy, parameter count, tokens/sec.
5. Ablation experiments over lags, fusion strategies, tied weights, and cell type.
6. Plots for training curves, final metrics, parameter-efficiency, and gate usage.
7. Text generation samples for qualitative inspection.
8. A conclusion-generation cell that turns the measured results into a compact report.

---

## Architecture in one equation

For layer $(\ell)$, branch $(j)$, and lag $(r_j)$:

$[h^{(\ell, j)}_t = \phi\left(W^{(\ell, j)}_x z^{(\ell-1)}_t + W^{(\ell, j)}_h h^{(\ell, j)}_{t-r_j} + b^{(\ell, j)}\right)]$

where $(r_1 = 1)$ is the normal recurrent branch and larger $(r_j)$ values are lagged branches.

The branch states are fused as:

$z^{(\ell)}_t = \text{Fuse}\left(h^{(\ell,1)}_t, h^{(\ell,2)}_t, \dots, h^{(\ell,m)}_t\right)$

Then $(z^{(\ell)}_t)$ becomes the input to layer $(\ell+1)$.

# 1. Research motivation

A vanilla RNN connects adjacent timesteps:

```text
h(t-100) -> h(t-99) -> ... -> h(t-1) -> h(t)
```

This means a distant token may require many recurrent transitions before it can influence the current prediction. That is one reason vanilla RNNs struggle with long-range dependencies.

A lagged branch creates a temporal shortcut:

```text
h(t-64) -------------------------------> h(t)
```

The proposed MS-LRNN keeps both:

```text
local branch:  h(t-1)  -> h(t)
lag branch:    h(t-r)  -> h(t)
```

The expected behavior is:

- `r = 1`: local syntax and immediate continuity;
- small lags like `r = 2, 4, 8`: phrase-scale and medium-distance patterns;
- large lags like `r = 16, 32, 64`: longer-range context and periodic structure.

This notebook tests whether that intuition helps on a real language modeling benchmark.

# 2. Dataset and metric choice

## Dataset

This notebook uses **WikiText-2**, a standard language-modeling benchmark built from Wikipedia articles. It is useful here because it preserves article-level text structure and therefore contains longer-range dependencies than toy sentence datasets.

The default Hugging Face dataset/config used below is:

```python
load_dataset("Salesforce/wikitext", "wikitext-2-v1")
```

## Why perplexity, not BLEU/ROUGE?

For **next-token language modeling**, the model outputs a probability distribution over the vocabulary at every timestep. The cleanest evaluation is therefore:

- cross-entropy / negative log-likelihood;
- perplexity = `exp(cross_entropy)`;
- token accuracy as a secondary sanity metric.

BLEU and ROUGE are more useful when a task has explicit reference texts, such as translation or summarization. For free-form language generation, many different continuations may be valid, so BLEU/ROUGE can be misleading.

In [ ]:
# Optional dependency bootstrap.
# In Colab/Kaggle/local environments, PyTorch is often already installed.
# This cell installs only missing packages.

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = [
    ("torch", "torch"),
    ("datasets", "datasets"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
]


def ensure_package(pip_name: str, import_name: str | None = None) -> None:
    import_name = import_name or pip_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for pip_name, import_name in REQUIRED_PACKAGES:
    ensure_package(pip_name, import_name)

In [ ]:
import copy
import dataclasses
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from matplotlib import pyplot as plt
from tqdm.auto import tqdm


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# 3. Run mode

The notebook has three run modes:

| mode | purpose | expected behavior |
|---|---|---|
| `quick` | sanity-check the full pipeline | small subset, small model, few epochs |
| `standard` | real WikiText-2 experiment | full WikiText-2, moderate model, more epochs |
| `full` | stronger research run | larger model, longer training, more ablations |

Start with `quick`. After confirming everything runs, switch to `standard` or `full`.

In [ ]:
@dataclass
class RunConfig:
    mode: str = "quick"
    dataset_name: str = "Salesforce/wikitext"
    dataset_config: str = "wikitext-2-v1"

    # Data limits. None means use the full split.
    train_line_limit: Optional[int] = 4_000
    valid_line_limit: Optional[int] = 600
    test_line_limit: Optional[int] = 600

    # Vocabulary.
    max_vocab_size: Optional[int] = 8_000
    min_freq: int = 1

    # BPTT/data layout.
    batch_size: int = 16
    seq_len: int = 64

    # Model defaults.
    emb_size: int = 64
    hidden_size: int = 64
    num_layers: int = 2
    dropout: float = 0.15
    layer_norm: bool = True
    tie_weights: bool = False

    # Optimization.
    epochs: int = 2
    lr: float = 2e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    patience: int = 3

    # Runtime limits for debugging. None means no artificial limit.
    max_train_batches: Optional[int] = 120
    max_eval_batches: Optional[int] = 60

    # Experiment control.
    seed: int = 42


MODE_CONFIGS = {
    "quick": RunConfig(
        mode="quick",
        train_line_limit=4_000,
        valid_line_limit=600,
        test_line_limit=600,
        max_vocab_size=8_000,
        batch_size=16,
        seq_len=64,
        emb_size=64,
        hidden_size=64,
        num_layers=2,
        dropout=0.15,
        epochs=2,
        max_train_batches=120,
        max_eval_batches=60,
    ),
    "standard": RunConfig(
        mode="standard",
        train_line_limit=None,
        valid_line_limit=None,
        test_line_limit=None,
        max_vocab_size=30_000,
        batch_size=32,
        seq_len=96,
        emb_size=128,
        hidden_size=128,
        num_layers=2,
        dropout=0.20,
        epochs=8,
        max_train_batches=None,
        max_eval_batches=None,
    ),
    "full": RunConfig(
        mode="full",
        train_line_limit=None,
        valid_line_limit=None,
        test_line_limit=None,
        max_vocab_size=None,
        batch_size=32,
        seq_len=128,
        emb_size=256,
        hidden_size=256,
        num_layers=3,
        dropout=0.25,
        epochs=20,
        lr=1e-3,
        max_train_batches=None,
        max_eval_batches=None,
    ),
}

# Change this to "standard" or "full" after the quick run succeeds.
RUN_MODE = "quick"
CFG = MODE_CONFIGS[RUN_MODE]
print(json.dumps(asdict(CFG), indent=2))

# 4. Load and tokenize WikiText-2

We use a simple word-level tokenizer based on whitespace. This makes the notebook transparent and easy to modify. For publishable comparisons against papers, keep the tokenization and vocabulary policy fixed across all models.

Important note: perplexity values are only directly comparable when the dataset, preprocessing, vocabulary, and context setup are the same.

In [ ]:
SPECIAL_TOKENS = ["<unk>", "<eos>"]
UNK_TOKEN = "<unk>"
EOS_TOKEN = "<eos>"
TOKEN_RE = re.compile(r"\S+")


def limit_split_texts(texts: Sequence[str], limit: Optional[int]) -> List[str]:
    texts = list(texts)
    return texts if limit is None else texts[:limit]


def tokenize_line(line: str) -> List[str]:
    tokens = TOKEN_RE.findall(line.strip())
    if not tokens:
        return []
    return tokens + [EOS_TOKEN]


def flatten_tokenized_lines(lines: Sequence[str]) -> List[str]:
    tokens: List[str] = []
    for line in lines:
        tokens.extend(tokenize_line(line))
    return tokens


def build_vocab(train_tokens: Sequence[str], max_vocab_size: Optional[int], min_freq: int) -> Tuple[Dict[str, int], List[str]]:
    counter = Counter(train_tokens)
    vocab = list(SPECIAL_TOKENS)
    special_set = set(vocab)

    # Frequency-first, then alphabetic for deterministic indices.
    candidates = sorted(counter.items(), key=lambda x: (-x[1], x[0]))
    for token, freq in candidates:
        if token in special_set:
            continue
        if freq < min_freq:
            continue
        vocab.append(token)
        if max_vocab_size is not None and len(vocab) >= max_vocab_size:
            break

    stoi = {token: idx for idx, token in enumerate(vocab)}
    return stoi, vocab


def encode_tokens(tokens: Sequence[str], stoi: Dict[str, int]) -> List[int]:
    unk_id = stoi[UNK_TOKEN]
    return [stoi.get(token, unk_id) for token in tokens]


raw = load_dataset(CFG.dataset_name, CFG.dataset_config)

train_lines = limit_split_texts(raw["train"]["text"], CFG.train_line_limit)
valid_lines = limit_split_texts(raw["validation"]["text"], CFG.valid_line_limit)
test_lines = limit_split_texts(raw["test"]["text"], CFG.test_line_limit)

train_tokens = flatten_tokenized_lines(train_lines)
valid_tokens = flatten_tokenized_lines(valid_lines)
test_tokens = flatten_tokenized_lines(test_lines)

stoi, itos = build_vocab(train_tokens, CFG.max_vocab_size, CFG.min_freq)
vocab_size = len(itos)

train_ids = torch.tensor(encode_tokens(train_tokens, stoi), dtype=torch.long)
valid_ids = torch.tensor(encode_tokens(valid_tokens, stoi), dtype=torch.long)
test_ids = torch.tensor(encode_tokens(test_tokens, stoi), dtype=torch.long)

summary = pd.DataFrame([
    {"split": "train", "lines": len(train_lines), "tokens": len(train_tokens)},
    {"split": "validation", "lines": len(valid_lines), "tokens": len(valid_tokens)},
    {"split": "test", "lines": len(test_lines), "tokens": len(test_tokens)},
])

display(summary)
print("Vocab size:", vocab_size)
print("First 30 vocab tokens:", itos[:30])
print("Example tokens:", train_tokens[:40])

In [ ]:
def batchify(token_ids: torch.Tensor, batch_size: int) -> torch.Tensor:
    """Arrange a 1D token stream into batch_size continuous streams.

    Output shape: [batch_size, num_steps]
    Each row is a contiguous stream. This is the classic PTB/WikiText language-modeling layout.
    """
    n_full = token_ids.numel() // batch_size
    token_ids = token_ids[: n_full * batch_size]
    return token_ids.view(batch_size, n_full).contiguous()


def get_batch(source: torch.Tensor, start: int, seq_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """Return input tokens and next-token targets.

    source shape: [batch_size, total_steps]
    output x/y shape: [batch_size, actual_seq_len]
    """
    actual_seq_len = min(seq_len, source.size(1) - 1 - start)
    x = source[:, start : start + actual_seq_len]
    y = source[:, start + 1 : start + 1 + actual_seq_len]
    return x, y


train_data = batchify(train_ids, CFG.batch_size).to(DEVICE)
valid_data = batchify(valid_ids, CFG.batch_size).to(DEVICE)
test_data = batchify(test_ids, CFG.batch_size).to(DEVICE)

print("train_data:", tuple(train_data.shape))
print("valid_data:", tuple(valid_data.shape))
print("test_data:", tuple(test_data.shape))

# 5. Visual intuition for the architecture

Each layer contains parallel recurrent branches. A branch with lag `r` reads the hidden state from `r` timesteps ago.

```text
Current input z(t)
      |
      +--> branch r=1:  h(t-1)  -> h(t)
      |
      +--> branch r=4:  h(t-4)  -> h(t)
      |
      +--> branch r=16: h(t-16) -> h(t)
      |
      +--> branch r=64: h(t-64) -> h(t)
      |
      v
   fusion/projection
      |
      v
next stacked layer
```

The code below implements this exactly and also supports a few controlled variations.

In [ ]:
def plot_lagged_recurrence(lags: Sequence[int] = (1, 4, 16), steps: int = 18) -> None:
    """Simple architecture plot showing temporal edges for chosen lags."""
    plt.figure(figsize=(12, 2.8))
    y_positions = np.linspace(0, len(lags) - 1, len(lags))
    for row, lag in enumerate(lags):
        y = y_positions[row]
        xs = np.arange(steps)
        plt.scatter(xs, np.full_like(xs, y, dtype=float), s=40)
        for t in range(lag, steps):
            plt.annotate(
                "",
                xy=(t, y),
                xytext=(t - lag, y),
                arrowprops=dict(arrowstyle="->", lw=1, shrinkA=3, shrinkB=3),
            )
        plt.text(-1.4, y, f"lag r={lag}", va="center")
    plt.yticks([])
    plt.xlabel("timestep")
    plt.title("Temporal recurrent edges for different lag branches")
    plt.tight_layout()
    plt.show()


plot_lagged_recurrence(lags=(1, 4, 8), steps=18)

# 6. Model implementation

Implementation details:

- `VanillaLaggedRNNCell`: \(\tanh(W_x x_t + W_h h_{t-r} + b)\)
- `GRULaggedCell`: same lag idea, but using a GRU update instead of vanilla tanh RNN
- `MultiScaleLaggedRNNLayer`: parallel lag branches + fusion
- `MSLaggedRNNLM`: embedding -> stacked lagged layers -> vocabulary decoder

The recurrent state is carried across truncated BPTT windows and detached between optimizer steps. That means the model can use lagged hidden states across chunk boundaries without backpropagating through the entire corpus.

In [ ]:
class VanillaLaggedRNNCell(nn.Module):
    """A simple tanh recurrent cell that consumes h_{t-r} instead of only h_{t-1}."""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.x2h = nn.Linear(input_size, hidden_size)
        self.h2h = nn.Linear(hidden_size, hidden_size, bias=False)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.xavier_uniform_(self.x2h.weight)
        nn.init.zeros_(self.x2h.bias)
        nn.init.orthogonal_(self.h2h.weight)

    def forward(self, x_t: torch.Tensor, h_lag: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.x2h(x_t) + self.h2h(h_lag))


class GRULaggedCell(nn.Module):
    """A GRUCell variant where the previous hidden state is h_{t-r}."""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.cell = nn.GRUCell(input_size, hidden_size)

    def forward(self, x_t: torch.Tensor, h_lag: torch.Tensor) -> torch.Tensor:
        return self.cell(x_t, h_lag)


def make_lagged_cell(cell_type: str, input_size: int, hidden_size: int) -> nn.Module:
    cell_type = cell_type.lower()
    if cell_type == "rnn":
        return VanillaLaggedRNNCell(input_size, hidden_size)
    if cell_type == "gru":
        return GRULaggedCell(input_size, hidden_size)
    raise ValueError(f"Unsupported cell_type={cell_type!r}. Expected 'rnn' or 'gru'.")


class MultiScaleLaggedRNNLayer(nn.Module):
    """One stacked recurrent layer containing multiple lagged recurrent branches.

    Parameters
    ----------
    input_size:
        Dimensionality of the input features z^{ell-1}_t.
    hidden_size:
        Dimensionality of each branch hidden state and of the fused layer output.
    lags:
        List such as [1, 4, 16]. Each branch j uses h_{t-r_j}.
    fusion:
        'concat_proj', 'sum', 'mean', 'gated', or 'learned_weighted_sum'.
    tied_branch_weights:
        If True, all lag branches share the same recurrent cell parameters. They still maintain separate hidden histories.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        lags: Sequence[int],
        fusion: str = "concat_proj",
        cell_type: str = "rnn",
        tied_branch_weights: bool = False,
        dropout: float = 0.0,
        layer_norm: bool = True,
    ):
        super().__init__()
        if not lags:
            raise ValueError("lags must be non-empty")
        if any(int(r) < 1 for r in lags):
            raise ValueError("all lags must be >= 1")

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.lags = tuple(int(r) for r in lags)
        self.num_branches = len(self.lags)
        self.max_lag = max(self.lags)
        self.fusion = fusion
        self.cell_type = cell_type
        self.tied_branch_weights = tied_branch_weights

        if tied_branch_weights:
            self.shared_cell = make_lagged_cell(cell_type, input_size, hidden_size)
            self.cells = None
        else:
            self.shared_cell = None
            self.cells = nn.ModuleList(
                [make_lagged_cell(cell_type, input_size, hidden_size) for _ in self.lags]
            )

        if fusion == "concat_proj":
            self.fusion_proj = nn.Linear(self.num_branches * hidden_size, hidden_size)
        elif fusion == "gated":
            self.gate = nn.Linear(self.num_branches * hidden_size, self.num_branches)
            self.gate_out = nn.Linear(hidden_size, hidden_size)
        elif fusion == "learned_weighted_sum":
            self.branch_logits = nn.Parameter(torch.zeros(self.num_branches))
        elif fusion in {"sum", "mean"}:
            pass
        else:
            raise ValueError(
                "fusion must be one of: concat_proj, gated, learned_weighted_sum, sum, mean"
            )

        self.norm = nn.LayerNorm(hidden_size) if layer_norm else nn.Identity()
        self.dropout = nn.Dropout(dropout)

    def _cell(self, branch_idx: int) -> nn.Module:
        if self.tied_branch_weights:
            return self.shared_cell
        assert self.cells is not None
        return self.cells[branch_idx]

    def initial_state(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
        # Shape: [batch, branches, max_lag, hidden]
        # state[:, j, -k, :] represents branch j's hidden state k timesteps before the current chunk.
        return torch.zeros(batch_size, self.num_branches, self.max_lag, self.hidden_size, device=device, dtype=dtype)

    def _fuse(self, branch_stack: torch.Tensor) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # branch_stack shape: [batch, branches, hidden]
        gate_alpha = None
        if self.fusion == "concat_proj":
            fused = self.fusion_proj(branch_stack.reshape(branch_stack.size(0), -1))
        elif self.fusion == "sum":
            fused = branch_stack.sum(dim=1) / math.sqrt(self.num_branches)
        elif self.fusion == "mean":
            fused = branch_stack.mean(dim=1)
        elif self.fusion == "learned_weighted_sum":
            alpha = torch.softmax(self.branch_logits, dim=0)
            fused = (branch_stack * alpha.view(1, -1, 1)).sum(dim=1)
            gate_alpha = alpha.expand(branch_stack.size(0), -1)
        elif self.fusion == "gated":
            flat = branch_stack.reshape(branch_stack.size(0), -1)
            gate_alpha = torch.softmax(self.gate(flat), dim=-1)
            fused = (branch_stack * gate_alpha.unsqueeze(-1)).sum(dim=1)
            fused = self.gate_out(fused)
        else:
            raise RuntimeError("unreachable")
        return fused, gate_alpha

    def forward(
        self,
        x: torch.Tensor,
        state: Optional[torch.Tensor] = None,
        collect_diagnostics: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
        # x shape: [batch, time, input_size]
        batch_size, time_steps, _ = x.shape
        if state is None:
            state = self.initial_state(batch_size, x.device, x.dtype)
        else:
            state = state.to(device=x.device, dtype=x.dtype)

        current_histories: List[List[torch.Tensor]] = [[] for _ in range(self.num_branches)]
        outputs: List[torch.Tensor] = []
        gate_history: List[torch.Tensor] = []

        for t in range(time_steps):
            branch_outputs: List[torch.Tensor] = []
            x_t = x[:, t, :]

            for branch_idx, lag in enumerate(self.lags):
                source_t = t - lag
                if source_t >= 0:
                    h_lag = current_histories[branch_idx][source_t]
                else:
                    # Convert source_t in [-max_lag, -1] to state index [0, max_lag-1].
                    state_idx = self.max_lag + source_t
                    h_lag = state[:, branch_idx, state_idx, :]

                h_t = self._cell(branch_idx)(x_t, h_lag)
                current_histories[branch_idx].append(h_t)
                branch_outputs.append(h_t)

            branch_stack = torch.stack(branch_outputs, dim=1)
            fused, alpha = self._fuse(branch_stack)
            fused = self.norm(fused)
            fused = self.dropout(fused)
            outputs.append(fused)

            if collect_diagnostics and alpha is not None:
                gate_history.append(alpha.detach())

        y = torch.stack(outputs, dim=1)

        next_state_branches: List[torch.Tensor] = []
        for branch_idx in range(self.num_branches):
            current_seq = torch.stack(current_histories[branch_idx], dim=1)
            combined = torch.cat([state[:, branch_idx, :, :], current_seq], dim=1)
            next_state_branches.append(combined[:, -self.max_lag :, :])
        next_state = torch.stack(next_state_branches, dim=1)

        diagnostics: Dict[str, torch.Tensor] = {}
        if collect_diagnostics and gate_history:
            diagnostics["gate_alpha"] = torch.stack(gate_history, dim=1)  # [batch, time, branches]

        return y, next_state, diagnostics


class MSLaggedRNNLM(nn.Module):
    """Embedding -> stacked MS-LRNN layers -> next-token vocabulary logits."""

    def __init__(
        self,
        vocab_size: int,
        emb_size: int,
        hidden_size: int,
        num_layers: int,
        lags: Sequence[int],
        fusion: str = "concat_proj",
        cell_type: str = "rnn",
        tied_branch_weights: bool = False,
        dropout: float = 0.1,
        layer_norm: bool = True,
        tie_weights: bool = False,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_size = emb_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lags = tuple(lags)
        self.fusion = fusion
        self.cell_type = cell_type
        self.tied_branch_weights = tied_branch_weights

        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.input_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList()

        for layer_idx in range(num_layers):
            layer_input_size = emb_size if layer_idx == 0 else hidden_size
            self.layers.append(
                MultiScaleLaggedRNNLayer(
                    input_size=layer_input_size,
                    hidden_size=hidden_size,
                    lags=lags,
                    fusion=fusion,
                    cell_type=cell_type,
                    tied_branch_weights=tied_branch_weights,
                    dropout=dropout,
                    layer_norm=layer_norm,
                )
            )

        self.output_dropout = nn.Dropout(dropout)
        self.decoder = nn.Linear(hidden_size, vocab_size)

        if tie_weights:
            if emb_size != hidden_size:
                raise ValueError("tie_weights=True requires emb_size == hidden_size")
            self.decoder.weight = self.embedding.weight

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        nn.init.zeros_(self.decoder.bias)

    def forward(
        self,
        input_ids: torch.Tensor,
        states: Optional[List[Optional[torch.Tensor]]] = None,
        collect_diagnostics: bool = False,
    ):
        # input_ids shape: [batch, time]
        x = self.input_dropout(self.embedding(input_ids))
        if states is None:
            states = [None] * len(self.layers)

        new_states: List[torch.Tensor] = []
        diagnostics: Dict[str, Dict[str, torch.Tensor]] = {}

        for layer_idx, layer in enumerate(self.layers):
            x, new_state, layer_diag = layer(
                x,
                state=states[layer_idx],
                collect_diagnostics=collect_diagnostics,
            )
            new_states.append(new_state)
            if layer_diag:
                diagnostics[f"layer_{layer_idx}"] = layer_diag

        logits = self.decoder(self.output_dropout(x))

        if collect_diagnostics:
            return logits, new_states, diagnostics
        return logits, new_states


def detach_states(states: Optional[List[torch.Tensor]]) -> Optional[List[torch.Tensor]]:
    if states is None:
        return None
    return [state.detach() if state is not None else None for state in states]


def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    params = model.parameters()
    if trainable_only:
        return sum(p.numel() for p in params if p.requires_grad)
    return sum(p.numel() for p in params)

In [ ]:
# Smoke test: verifies tensor shapes, recurrent states, and loss computation before training.

def smoke_test_model() -> None:
    model = MSLaggedRNNLM(
        vocab_size=101,
        emb_size=16,
        hidden_size=16,
        num_layers=2,
        lags=[1, 4, 8],
        fusion="gated",
        cell_type="rnn",
        dropout=0.0,
    ).to(DEVICE)
    x = torch.randint(0, 101, (3, 12), device=DEVICE)
    y = torch.randint(0, 101, (3, 12), device=DEVICE)
    logits, states, diagnostics = model(x, collect_diagnostics=True)
    loss = F.cross_entropy(logits.reshape(-1, 101), y.reshape(-1))
    loss.backward()
    print("logits:", tuple(logits.shape))
    print("num states:", len(states), "state[0]:", tuple(states[0].shape))
    print("diagnostic layers:", list(diagnostics.keys()))
    print("loss:", float(loss.detach().cpu()))
    print("parameters:", count_parameters(model))


smoke_test_model()

# 7. Metrics and training utilities

For every experiment we track:

- `train_loss`: average cross-entropy on training chunks;
- `val_loss`: validation cross-entropy;
- `val_ppl`: validation perplexity;
- `val_acc`: token-level top-1 accuracy;
- `grad_norm`: gradient norm before clipping;
- `tokens_per_sec`: rough throughput;
- `n_params`: trainable parameter count.

For final reporting, the best checkpoint by validation perplexity is restored and evaluated on the test split.

In [ ]:
def safe_exp(x: float) -> float:
    return float(math.exp(x)) if x < 50 else float("inf")


def lm_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    vocab_size = logits.size(-1)
    return F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))


def token_accuracy(logits: torch.Tensor, targets: torch.Tensor) -> Tuple[int, int]:
    preds = logits.argmax(dim=-1)
    correct = (preds == targets).sum().item()
    total = targets.numel()
    return int(correct), int(total)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    data: torch.Tensor,
    seq_len: int,
    max_batches: Optional[int] = None,
) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    states = None
    start_time = time.time()

    starts = range(0, data.size(1) - 1, seq_len)
    for batch_idx, start in enumerate(starts):
        if max_batches is not None and batch_idx >= max_batches:
            break
        x, y = get_batch(data, start, seq_len)
        logits, states = model(x, states=states)
        states = detach_states(states)
        loss = lm_loss(logits, y)
        correct, n_tokens = token_accuracy(logits, y)
        total_loss += loss.item() * n_tokens
        total_correct += correct
        total_tokens += n_tokens

    elapsed = max(time.time() - start_time, 1e-9)
    avg_loss = total_loss / max(total_tokens, 1)
    return {
        "loss": avg_loss,
        "ppl": safe_exp(avg_loss),
        "bpt": avg_loss / math.log(2),
        "acc": total_correct / max(total_tokens, 1),
        "tokens": total_tokens,
        "tokens_per_sec": total_tokens / elapsed,
    }


@dataclass
class ExperimentConfig:
    name: str
    lags: Tuple[int, ...]
    fusion: str = "concat_proj"
    cell_type: str = "rnn"
    tied_branch_weights: bool = False
    emb_size: Optional[int] = None
    hidden_size: Optional[int] = None
    num_layers: Optional[int] = None
    dropout: Optional[float] = None
    lr: Optional[float] = None
    seed: int = 42


def make_model(exp: ExperimentConfig, cfg: RunConfig, vocab_size: int) -> MSLaggedRNNLM:
    emb_size = exp.emb_size or cfg.emb_size
    hidden_size = exp.hidden_size or cfg.hidden_size
    num_layers = exp.num_layers or cfg.num_layers
    dropout = cfg.dropout if exp.dropout is None else exp.dropout

    model = MSLaggedRNNLM(
        vocab_size=vocab_size,
        emb_size=emb_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        lags=exp.lags,
        fusion=exp.fusion,
        cell_type=exp.cell_type,
        tied_branch_weights=exp.tied_branch_weights,
        dropout=dropout,
        layer_norm=cfg.layer_norm,
        tie_weights=cfg.tie_weights,
    )
    return model


def train_one_experiment(
    exp: ExperimentConfig,
    cfg: RunConfig,
    train_data: torch.Tensor,
    valid_data: torch.Tensor,
    test_data: torch.Tensor,
    vocab_size: int,
) -> Tuple[MSLaggedRNNLM, pd.DataFrame, Dict[str, Any]]:
    set_seed(exp.seed)
    model = make_model(exp, cfg, vocab_size).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=exp.lr or cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    n_params = count_parameters(model)
    best_val_loss = float("inf")
    best_state_dict = None
    bad_epochs = 0
    history: List[Dict[str, Any]] = []

    print(f"\n=== Experiment: {exp.name} ===")
    print("lags:", exp.lags, "fusion:", exp.fusion, "cell:", exp.cell_type,
          "tied:", exp.tied_branch_weights, "params:", f"{n_params:,}")

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        total_correct = 0
        grad_norms: List[float] = []
        states = None
        start_time = time.time()

        starts = list(range(0, train_data.size(1) - 1, cfg.seq_len))
        progress = tqdm(starts, desc=f"{exp.name} epoch {epoch}/{cfg.epochs}", leave=False)
        for batch_idx, start in enumerate(progress):
            if cfg.max_train_batches is not None and batch_idx >= cfg.max_train_batches:
                break

            x, y = get_batch(train_data, start, cfg.seq_len)
            optimizer.zero_grad(set_to_none=True)
            logits, states = model(x, states=states)
            loss = lm_loss(logits, y)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            states = detach_states(states)

            correct, n_tokens = token_accuracy(logits.detach(), y)
            total_loss += loss.item() * n_tokens
            total_correct += correct
            total_tokens += n_tokens
            grad_norms.append(float(grad_norm.detach().cpu()) if torch.is_tensor(grad_norm) else float(grad_norm))

            progress.set_postfix({
                "loss": total_loss / max(total_tokens, 1),
                "ppl": safe_exp(total_loss / max(total_tokens, 1)),
            })

        elapsed = max(time.time() - start_time, 1e-9)
        train_loss = total_loss / max(total_tokens, 1)
        val = evaluate(model, valid_data, cfg.seq_len, max_batches=cfg.max_eval_batches)

        row = {
            "experiment": exp.name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_ppl": safe_exp(train_loss),
            "train_acc": total_correct / max(total_tokens, 1),
            "val_loss": val["loss"],
            "val_ppl": val["ppl"],
            "val_bpt": val["bpt"],
            "val_acc": val["acc"],
            "grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else np.nan,
            "tokens_per_sec_train": total_tokens / elapsed,
            "tokens_per_sec_val": val["tokens_per_sec"],
            "n_params": n_params,
            "lags": str(exp.lags),
            "fusion": exp.fusion,
            "cell_type": exp.cell_type,
            "tied_branch_weights": exp.tied_branch_weights,
        }
        history.append(row)

        print(
            f"epoch {epoch:02d} | "
            f"train ppl {row['train_ppl']:.2f} | "
            f"val ppl {row['val_ppl']:.2f} | "
            f"val acc {row['val_acc']:.3f} | "
            f"tok/s {row['tokens_per_sec_train']:.0f}"
        )

        if val["loss"] < best_val_loss:
            best_val_loss = val["loss"]
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.patience:
                print(f"Early stopping after {bad_epochs} non-improving epochs.")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        model.to(DEVICE)

    val_final = evaluate(model, valid_data, cfg.seq_len, max_batches=cfg.max_eval_batches)
    test_final = evaluate(model, test_data, cfg.seq_len, max_batches=cfg.max_eval_batches)

    final = {
        "experiment": exp.name,
        "best_val_loss": val_final["loss"],
        "best_val_ppl": val_final["ppl"],
        "best_val_acc": val_final["acc"],
        "test_loss": test_final["loss"],
        "test_ppl": test_final["ppl"],
        "test_acc": test_final["acc"],
        "n_params": n_params,
        "lags": str(exp.lags),
        "fusion": exp.fusion,
        "cell_type": exp.cell_type,
        "tied_branch_weights": exp.tied_branch_weights,
        "seed": exp.seed,
    }
    return model, pd.DataFrame(history), final

# 8. Ablation design

We compare:

1. **Vanilla RNN baseline**: only `lags=[1]`.
2. **Two-branch lagged model**: `lags=[1, 4]`.
3. **Multi-scale lagged model**: `lags=[1, 4, 16]`.
4. **Gated multi-scale model**: learns per-token weights over lag branches.
5. **Tied branch weights**: all lag branches share the same RNN cell parameters while keeping separate histories.
6. **GRU lagged model**: replaces tanh recurrence with GRU recurrence.

For a quick run, the notebook uses the first few experiments only. For `standard` and `full`, it runs more variants.

In [ ]:
ALL_EXPERIMENTS = [
    ExperimentConfig(
        name="baseline_rnn_lag1",
        lags=(1,),
        fusion="mean",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="two_branch_rnn_lags_1_4",
        lags=(1, 4),
        fusion="concat_proj",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="multi_rnn_lags_1_4_16",
        lags=(1, 4, 16),
        fusion="concat_proj",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="gated_rnn_lags_1_4_16",
        lags=(1, 4, 16),
        fusion="gated",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="tied_rnn_lags_1_4_16",
        lags=(1, 4, 16),
        fusion="concat_proj",
        cell_type="rnn",
        tied_branch_weights=True,
    ),
    ExperimentConfig(
        name="multi_rnn_lags_1_2_4_8",
        lags=(1, 2, 4, 8),
        fusion="concat_proj",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="multi_rnn_lags_1_4_16_64",
        lags=(1, 4, 16, 64),
        fusion="concat_proj",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="gated_rnn_lags_1_4_16_64",
        lags=(1, 4, 16, 64),
        fusion="gated",
        cell_type="rnn",
    ),
    ExperimentConfig(
        name="gru_lags_1_4_16",
        lags=(1, 4, 16),
        fusion="concat_proj",
        cell_type="gru",
    ),
]

if CFG.mode == "quick":
    EXPERIMENTS = ALL_EXPERIMENTS[:4]
elif CFG.mode == "standard":
    EXPERIMENTS = ALL_EXPERIMENTS[:7]
else:
    EXPERIMENTS = ALL_EXPERIMENTS

pd.DataFrame([dataclasses.asdict(e) for e in EXPERIMENTS])

# 9. Run the experiments

This is the main benchmark loop. In `quick` mode it is designed to finish relatively quickly. In `standard` or `full`, it may take much longer depending on your GPU/CPU.

In [ ]:
RUN_BENCHMARK = True

trained_models: Dict[str, MSLaggedRNNLM] = {}
histories: List[pd.DataFrame] = []
final_rows: List[Dict[str, Any]] = []

if RUN_BENCHMARK:
    for exp in EXPERIMENTS:
        model, hist, final = train_one_experiment(
            exp=exp,
            cfg=CFG,
            train_data=train_data,
            valid_data=valid_data,
            test_data=test_data,
            vocab_size=vocab_size,
        )
        trained_models[exp.name] = model
        histories.append(hist)
        final_rows.append(final)

        # Keep memory under control when running many models.
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

history_df = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
results_df = pd.DataFrame(final_rows).sort_values("best_val_ppl") if final_rows else pd.DataFrame()

display(results_df)

# 10. Training curves and metric plots

The plots below are intentionally simple matplotlib figures so they export cleanly to PDF/HTML.

In [ ]:
def plot_history_metric(history_df: pd.DataFrame, metric: str, ylabel: str, logy: bool = False) -> None:
    if history_df.empty:
        print("No history to plot.")
        return
    plt.figure(figsize=(9, 5))
    for name, group in history_df.groupby("experiment"):
        plt.plot(group["epoch"], group[metric], marker="o", label=name)
    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    plt.title(ylabel + " over training")
    if logy:
        plt.yscale("log")
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_history_metric(history_df, "train_ppl", "Train perplexity", logy=True)
plot_history_metric(history_df, "val_ppl", "Validation perplexity", logy=True)
plot_history_metric(history_df, "val_acc", "Validation token accuracy", logy=False)

In [ ]:
def bar_plot_results(results_df: pd.DataFrame, metric: str, title: str, ylabel: str) -> None:
    if results_df.empty:
        print("No results to plot.")
        return
    df = results_df.sort_values(metric, ascending=True)
    plt.figure(figsize=(10, 5))
    plt.bar(df["experiment"], df[metric])
    plt.xticks(rotation=35, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


bar_plot_results(results_df, "best_val_ppl", "Best validation perplexity by experiment", "perplexity")
bar_plot_results(results_df, "test_ppl", "Test perplexity by experiment", "perplexity")
bar_plot_results(results_df, "n_params", "Trainable parameter count by experiment", "parameters")

In [ ]:
def scatter_efficiency(results_df: pd.DataFrame) -> None:
    if results_df.empty:
        print("No results to plot.")
        return
    plt.figure(figsize=(8, 5))
    plt.scatter(results_df["n_params"], results_df["test_ppl"], s=80)
    for _, row in results_df.iterrows():
        plt.annotate(row["experiment"], (row["n_params"], row["test_ppl"]), fontsize=8, alpha=0.8)
    plt.xlabel("trainable parameters")
    plt.ylabel("test perplexity")
    plt.title("Parameter efficiency: lower-left is better")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


scatter_efficiency(results_df)

# 11. Gating diagnostics

For experiments using `fusion="gated"`, the model learns a per-token softmax over branches. This lets us inspect whether the model prefers short or long lags.

Interpretation caution: gate weights are not a perfect causal explanation, but they are useful diagnostics.

In [ ]:
@torch.no_grad()
def collect_gate_usage(
    model: MSLaggedRNNLM,
    data: torch.Tensor,
    seq_len: int,
    max_batches: int = 10,
) -> Dict[str, np.ndarray]:
    model.eval()
    states = None
    accum: Dict[str, List[np.ndarray]] = {}

    for batch_idx, start in enumerate(range(0, data.size(1) - 1, seq_len)):
        if batch_idx >= max_batches:
            break
        x, _ = get_batch(data, start, seq_len)
        logits, states, diagnostics = model(x, states=states, collect_diagnostics=True)
        states = detach_states(states)
        for layer_name, layer_diag in diagnostics.items():
            if "gate_alpha" in layer_diag:
                # [batch, time, branches] -> average over batch/time
                avg = layer_diag["gate_alpha"].mean(dim=(0, 1)).detach().cpu().numpy()
                accum.setdefault(layer_name, []).append(avg)

    return {name: np.stack(values).mean(axis=0) for name, values in accum.items()}


def plot_gate_usage(model: MSLaggedRNNLM, usage: Dict[str, np.ndarray]) -> None:
    if not usage:
        print("No gate usage found. Select a gated model.")
        return
    lags = list(model.lags)
    for layer_name, values in usage.items():
        plt.figure(figsize=(6, 4))
        plt.bar([str(r) for r in lags], values)
        plt.ylim(0, 1)
        plt.xlabel("lag r")
        plt.ylabel("mean gate weight")
        plt.title(f"Mean branch gate usage: {layer_name}")
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


if trained_models:
    gated_names = [name for name, model in trained_models.items() if model.fusion == "gated"]
    if gated_names:
        gated_name = gated_names[0]
        print("Inspecting gated model:", gated_name)
        usage = collect_gate_usage(trained_models[gated_name], valid_data, CFG.seq_len, max_batches=10)
        plot_gate_usage(trained_models[gated_name], usage)
    else:
        print("No gated model was trained in this run.")

# 12. Text generation samples

These samples are qualitative only. A model can have decent perplexity and still generate awkward text, especially at small scale. Use this section to sanity-check whether the model has learned local continuity.

In [ ]:
def tokenize_prompt(prompt: str) -> List[str]:
    toks = TOKEN_RE.findall(prompt.strip())
    return toks if toks else [EOS_TOKEN]


def decode_ids(ids: Sequence[int]) -> str:
    return " ".join(itos[int(i)] for i in ids)


@torch.no_grad()
def generate_text(
    model: MSLaggedRNNLM,
    prompt: str,
    max_new_tokens: int = 80,
    temperature: float = 0.9,
    top_k: int = 40,
) -> str:
    model.eval()
    prompt_ids = [stoi.get(tok, stoi[UNK_TOKEN]) for tok in tokenize_prompt(prompt)]
    generated = list(prompt_ids)
    states = None

    # Prime the recurrent states with the prompt.
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    logits, states = model(x, states=states)
    next_logits = logits[:, -1, :]

    for _ in range(max_new_tokens):
        logits_1d = next_logits.squeeze(0) / max(temperature, 1e-6)
        if top_k is not None and top_k > 0:
            k = min(top_k, logits_1d.numel())
            values, indices = torch.topk(logits_1d, k=k)
            probs = torch.softmax(values, dim=-1)
            sampled_relative = torch.multinomial(probs, num_samples=1)
            next_id = indices[sampled_relative].item()
        else:
            probs = torch.softmax(logits_1d, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

        generated.append(next_id)
        x = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)
        logits, states = model(x, states=states)
        states = detach_states(states)
        next_logits = logits[:, -1, :]

    return decode_ids(generated)


if trained_models and not results_df.empty:
    best_name = results_df.sort_values("best_val_ppl").iloc[0]["experiment"]
    best_model = trained_models[best_name]
    print("Best model by validation perplexity:", best_name)
    for prompt in ["The history of", "In the early", "The city was"]:
        print("\nPROMPT:", prompt)
        print(generate_text(best_model, prompt, max_new_tokens=60, temperature=0.9, top_k=50))

# 13. Optional: repeated-seed robustness

A single run is not enough for a publishable claim. Once a promising configuration emerges, rerun it with several seeds and report mean ± standard deviation.

The function below is intentionally optional because it multiplies compute cost.

In [ ]:
def run_seed_sweep(
    base_exp: ExperimentConfig,
    seeds: Sequence[int] = (11, 22, 33),
) -> pd.DataFrame:
    rows = []
    for seed in seeds:
        exp = dataclasses.replace(base_exp, name=f"{base_exp.name}_seed_{seed}", seed=seed)
        model, hist, final = train_one_experiment(
            exp=exp,
            cfg=CFG,
            train_data=train_data,
            valid_data=valid_data,
            test_data=test_data,
            vocab_size=vocab_size,
        )
        rows.append(final)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        gc.collect()
    return pd.DataFrame(rows)


RUN_SEED_SWEEP = False

if RUN_SEED_SWEEP and not results_df.empty:
    best_name = results_df.sort_values("best_val_ppl").iloc[0]["experiment"]
    best_exp = next(exp for exp in EXPERIMENTS if exp.name == best_name)
    seed_df = run_seed_sweep(best_exp, seeds=(11, 22, 33))
    display(seed_df)
    print(seed_df[["best_val_ppl", "test_ppl", "test_acc"]].agg(["mean", "std"]))

# 14. Automated conclusion report

Run this after the benchmark. It summarizes which architecture won and whether the lagged idea helped relative to the vanilla `lags=[1]` baseline.

In [ ]:
def make_conclusion(results_df: pd.DataFrame) -> str:
    if results_df.empty:
        return "No benchmark results are available yet. Run the benchmark cell first."

    df = results_df.sort_values("best_val_ppl").reset_index(drop=True)
    best = df.iloc[0]
    baseline_rows = df[df["experiment"] == "baseline_rnn_lag1"]
    lines = []
    lines.append("# Experiment conclusion")
    lines.append("")
    lines.append(
        f"Best validation model: **{best['experiment']}** with validation perplexity "
        f"**{best['best_val_ppl']:.2f}** and test perplexity **{best['test_ppl']:.2f}**."
    )
    lines.append(
        f"It used lags `{best['lags']}`, fusion `{best['fusion']}`, cell type `{best['cell_type']}`, "
        f"and `{int(best['n_params']):,}` trainable parameters."
    )

    if not baseline_rows.empty:
        baseline = baseline_rows.iloc[0]
        delta = baseline["test_ppl"] - best["test_ppl"]
        rel = 100 * delta / baseline["test_ppl"] if baseline["test_ppl"] else float("nan")
        lines.append("")
        if delta > 0:
            lines.append(
                f"Compared with the vanilla RNN baseline, the best model reduced test perplexity by "
                f"**{delta:.2f}** points (**{rel:.1f}%** relative improvement)."
            )
        else:
            lines.append(
                f"Compared with the vanilla RNN baseline, the best model did not improve test perplexity "
                f"in this run; the baseline was better by **{-delta:.2f}** points."
            )

    lines.append("")
    lines.append("Interpretation guidance:")
    lines.append("- If multi-lag variants beat `lags=[1]`, that supports the hypothesis that explicit temporal shortcuts help.")
    lines.append("- If gated fusion wins, branch selection is useful and different lags may specialize.")
    lines.append("- If tied branch weights are competitive, the architecture may benefit from lag diversity without many extra parameters.")
    lines.append("- For a publishable claim, rerun the top 2-3 models with several seeds and a full training budget.")
    return "\n".join(lines)


print(make_conclusion(results_df))

# 15. Extensions worth trying

## A. Top-down recurrent feedback

The current notebook implements lagged recurrence inside each layer. You also asked whether RNNs can go back to previous-layer neurons. Yes: one can add top-down feedback from higher layers into lower layers at the next timestep.

For example:

\[
h^{(1)}_t = f(W_x x_t + U h^{(1)}_{t-1} + V h^{(2)}_{t-1} + b)
\]

where layer 1 receives feedback from layer 2's previous timestep.

That is not included in the main benchmark because it changes the architecture more aggressively and makes causality/scheduling more complex. A clean next notebook could compare:

1. vanilla stacked RNN;
2. MS-LRNN;
3. top-down feedback RNN;
4. MS-LRNN + top-down feedback.

## B. Learnable lags

Instead of fixed `lags=[1,4,16]`, one could learn a soft attention over a memory bank of previous hidden states.

## C. Hybrid with attention

A practical modern variant is:

```text
Embedding -> MS-LRNN blocks -> lightweight causal attention -> decoder
```

This would test whether lagged recurrence can serve as a cheap local/medium-range temporal processor before attention.

# 16. References and reproducibility notes

- WikiText dataset card: https://huggingface.co/datasets/Salesforce/wikitext
- WikiText paper/citation from the dataset card: Stephen Merity, Caiming Xiong, James Bradbury, Richard Socher, *Pointer Sentinel Mixture Models*, 2016.
- PyTorch documentation: https://docs.pytorch.org/
- TorchMetrics perplexity explanation: https://lightning.ai/docs/torchmetrics/stable/gallery/text/perplexity.html

Reproducibility checklist:

1. Keep tokenization fixed across all experiments.
2. Keep the same train/validation/test split.
3. Keep the same sequence length and batch size unless explicitly studying those factors.
4. Report parameter count and training budget.
5. Use validation perplexity for model selection and test perplexity for final reporting.
6. Repeat final candidates with multiple random seeds.